# ML-04 — Week 03 Data Contract & Leakage Check

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring

This notebook uses the warehouse release and develops only on `month=2026-03`. The June `_sample` is treated as sealed test data.


## 1) Contract — plain words

**1. What one row means:** One row is one content page for one client on one report date (`report_date + client_hash_id + content_hash_id`).

**2. Tables:** `fact_content_daily_performance` is the primary table. I join `dim_content` only if I need page metadata; `dim_clients` is context for client history/access checks.

**3. Time window:** March 2026 is the feature window. The decision moment is the end of March. April 2026 is the outcome window used only to create the development label. June 2026 is sealed and never used to develop the label or features.

**4. What I predict/rank:** I rank pages for refresh review using a forward proxy: `future_decline_label = 1` when April GSC impressions are at least 20% lower than March impressions. This is a review-priority proxy, not a causal claim that refreshing a page will recover it.

**5. Deliberate exclusion:** I exclude April outcome fields from the honest feature frame because they are unknown at the March decision moment. I also exclude IDs as model features.


## 2) Data rules

- IDs are grouping/join keys only.
- Search/analytics availability flags are three-valued; use `IS TRUE`, never `= TRUE` or `NOT flag`.
- `gsc_avg_position = 0` means no position data, not rank zero.
- Do not use the June `_sample` to develop labels or features.


In [ ]:
%pip -q install duckdb huggingface_hub pandas scikit-learn

import os, duckdb, pandas as pd
from getpass import getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = None
if not HF_TOKEN:
    raise RuntimeError('HF_TOKEN is missing. In Colab, add it as a Secret named HF_TOKEN; never paste it into the notebook.')

con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf_secret (TYPE huggingface, TOKEN ?)", [HF_TOKEN])
BASE = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"{BASE}/fact_content_daily_performance/**/*.parquet"
print('DuckDB + Hugging Face connection ready.')


## 3) Three verification queries — exactly three

### Query 1 — grain
A valid daily fact should have no duplicate `(report_date, client_hash_id, content_hash_id)` keys in the March partition.


In [ ]:
q1 = """
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS row_count
FROM read_parquet(?)
WHERE month = '2026-03'
GROUP BY 1,2,3
HAVING COUNT(*) > 1
LIMIT 5
"""
grain_violations = con.sql(q1, params=[FACT]).df()
print(f'Grain violations: {len(grain_violations)}')
display(grain_violations)


### Query 2 — row count and date span
This establishes the actual March slice rather than relying on the documentation alone.


In [ ]:
feature_sql = """
WITH march AS (
  SELECT *
  FROM read_parquet(?)
  WHERE month = '2026-03'
    AND gsc_data_available IS TRUE
), agg AS (
  SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions_march,
    SUM(gsc_clicks) AS clicks_march,
    AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_march,
    100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr_pct_march,
    COUNT(DISTINCT report_date) AS observed_days_march
  FROM march
  GROUP BY 1,2
), april AS (
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS impressions_april
  FROM read_parquet(?)
  WHERE month = '2026-04'
    AND gsc_data_available IS TRUE
  GROUP BY 1,2
)
SELECT a.*, b.impressions_april,
       CASE WHEN b.impressions_april IS NOT NULL
                 AND a.impressions_march > 0
                 AND b.impressions_april < 0.80 * a.impressions_march
            THEN 1 ELSE 0 END AS future_decline_label
FROM agg a
LEFT JOIN april b USING (client_hash_id, content_hash_id)
WHERE b.impressions_april IS NOT NULL
"""
features = con.sql(feature_sql, params=[FACT, FACT]).df()
print(f'Feature/label rows: {len(features):,}')
display(features.head(10))


### Feature availability notes

- **`impressions_march`** — knowable at the decision moment because it is the March GSC impression total already observed.
- **`clicks_march`** — knowable at the decision moment because March GSC clicks have already been recorded.
- **`avg_position_march`** — knowable at the decision moment because it summarizes observed March GSC positions; zero-position records are ignored.
- **`ctr_pct_march`** — knowable at the decision moment because it is computed only from March clicks and impressions.
- **`observed_days_march`** — knowable at the decision moment because it counts the March report dates actually present for the page.


In [ ]:
q3 = """
SELECT COUNT(*) AS march_rows_with_gsc_available
FROM read_parquet(?)
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
"""
gsc_available = con.sql(q3, params=[FACT]).df()
display(gsc_available)


# Deliberately leak the label itself. This should make the quick score perfect or effectively perfect.
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

honest_cols = ['impressions_march','clicks_march','avg_position_march','ctr_pct_march','observed_days_march']
X = features[honest_cols].fillna(0)
y = features['future_decline_label']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
honest_model = RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1, class_weight='balanced')
honest_model.fit(Xtr, ytr)
honest_auc = roc_auc_score(yte, honest_model.predict_proba(Xte)[:,1])

X_leak = features[honest_cols].copy()
X_leak['future_decline_label'] = y  # INTENTIONAL LEAK
Xtr, Xte, ytr, yte = train_test_split(X_leak, y, test_size=0.25, random_state=42, stratify=y)
leaky_model = RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1, class_weight='balanced')
leaky_model.fit(Xtr, ytr)
leaky_auc = roc_auc_score(yte, leaky_model.predict_proba(Xte)[:,1])

print(f'Honest ROC-AUC: {honest_auc:.4f}')
print(f'Leaky ROC-AUC:  {leaky_auc:.4f}')
print(f'Leakage lift:   {leaky_auc - honest_auc:+.4f}')


In [ ]:
# REMOVE the leaked column and retain only the honest March features.
HONEST_FEATURES = ['impressions_march','clicks_march','avg_position_march','ctr_pct_march','observed_days_march']
features_honest = features[HONEST_FEATURES].copy()
assert 'future_decline_label' not in features_honest.columns
assert 'impressions_april' not in features_honest.columns
print('Final honest feature frame columns:')
print(list(features_honest.columns))


### Feature availability notes

- **`impressions_31d`** — knowable at the decision moment because it is the March GSC impression total observed before the review queue is built.
- **`clicks_31d`** — knowable at the decision moment because it is the March GSC click total already recorded by the decision date.
- **`avg_position`** — knowable at the decision moment because it summarizes March GSC positions already observed; zero-position records are excluded from the mean.
- **`ctr_pct`** — knowable at the decision moment because it is calculated only from March clicks and impressions, both already observed.
- **`observed_days`** — knowable at the decision moment because it counts the March report dates actually present for the page, giving a simple coverage signal.


## 3B) Deliberate leakage experiment

The experiment intentionally adds a label-derived column. The label is `1` when `trend_direction = 'down'`. Because `trend_direction` is effectively the label definition, a model using it should look unrealistically strong. The point is to demonstrate the trap, not to keep the result.


In [ ]:
# Pull only the columns needed for the controlled leakage demonstration.
leak_sql = """
WITH march AS (
  SELECT client_hash_id, content_hash_id, report_date,
         gsc_impressions, gsc_clicks, gsc_avg_position,
         trend_direction
  FROM read_parquet(?)
  WHERE month = '2026-03'
    AND gsc_data_available IS TRUE
), page AS (
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS impressions_31d,
         SUM(gsc_clicks) AS clicks_31d,
         AVG(NULLIF(gsc_avg_position, 0)) AS avg_position,
         100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr_pct,
         ANY_VALUE(trend_direction) AS trend_direction
  FROM march
  GROUP BY 1,2
)
SELECT *, (trend_direction = 'down')::INTEGER AS is_declining_label
FROM page
WHERE trend_direction IS NOT NULL
"""
leak_df = con.sql(leak_sql, params=[FACT]).df()

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

base_cols = ['impressions_31d','clicks_31d','avg_position','ctr_pct']
X = leak_df[base_cols].fillna(0)
y = leak_df['is_declining_label']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
honest_model = RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1, class_weight='balanced')
honest_model.fit(Xtr, ytr)
honest_auc = roc_auc_score(yte, honest_model.predict_proba(Xte)[:,1])

X_leak = leak_df[base_cols + ['trend_direction']].copy()
X_leak['trend_direction'] = (X_leak['trend_direction'] == 'down').astype(int)
Xtr, Xte, ytr, yte = train_test_split(X_leak, y, test_size=0.25, random_state=42, stratify=y)
leaky_model = RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1, class_weight='balanced')
leaky_model.fit(Xtr, ytr)
leaky_auc = roc_auc_score(yte, leaky_model.predict_proba(Xte)[:,1])

print(f'Honest ROC-AUC: {honest_auc:.4f}')
print(f'Leaky ROC-AUC:  {leaky_auc:.4f}')
print(f'Leakage lift:   {leaky_auc - honest_auc:+.4f}')


In [ ]:
# REMOVE the leaked column and retain only the honest feature set.
HONEST_FEATURES = ['impressions_31d','clicks_31d','avg_position','ctr_pct','observed_days']
features_honest = features[HONEST_FEATURES].copy()
assert 'trend_direction' not in features_honest.columns
assert 'is_declining_label' not in features_honest.columns
print('Final honest feature frame columns:')
print(list(features_honest.columns))


## 4) Named limitation

**Limitation:** March is an unbalanced panel. Different clients have different tracking start dates, so a page missing early observations may have less opportunity to accumulate impressions or clicks. The March slice therefore needs coverage checks before any later model comparison, and this Week 03 frame should not be treated as a causal estimate of which pages will recover after a refresh.


## 5) Self-check

- [ ] Contract states the row grain, tables, time window, target/proxy, and one exclusion.
- [ ] Exactly three verification queries are shown; availability uses `IS TRUE`.
- [ ] Five features are shown, with an availability-at-decision explanation for each.
- [ ] Leakage experiment is visible, then the leaked column is removed from the retained feature frame.
- [ ] June `_sample` was not used for feature/label development.
- [ ] No HF token appears in the notebook.
- [ ] Only this notebook is changed for the Week 03 submission.
